# Columbus crime - a worked replication

This notebook reproduces the analysis that opens Anselin's *Spatial Econometrics:
Methods and Models* (1988), using the dataset that has served as the field's
teaching example ever since.

**The data.** 49 contiguous planning neighbourhoods in Columbus, Ohio, 1980.
Residential burglaries and vehicle thefts per thousand households (`CRIME`),
household income in $1000s (`INC`), and housing value in $1000s (`HOVAL`).
The file in `data/` was taken from the
[libpysal](https://github.com/pysal/libpysal) example collection, which carries
Anselin's original values; `columbus.gal` is the matching first-order contiguity
structure in GeoDa's GAL format.

**What the notebook shows.** That OLS is the wrong model here, how the Lagrange
Multiplier battery says *which* spatial model to fit instead, and why the
coefficients of a spatial lag model are not its marginal effects.

> **Citation.** Anselin, L. (1988). *Spatial Econometrics: Methods and Models*.
> Dordrecht: Kluwer Academic.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

using SpatialEconometrics
using LinearAlgebra, SparseArrays, Statistics, Printf
import StatsAPI: coef, coefnames, stderror, loglikelihood, nobs, residuals

## 1. Loading the data

No CSV package is needed for 49 rows, and writing the reader out makes the data
layout explicit. `read_gal` is the package's reader for GeoDa contiguity files;
passing `ids` aligns the weight matrix with the row order of the table rather
than trusting the order the file happens to use.

In [ ]:
function read_columbus(dir)
    lines = readlines(joinpath(dir, "columbus.csv"))
    hdr   = split(strip(lines[1]), ',')
    vals  = [parse.(Float64, split(strip(l), ',')) for l in lines[2:end]]
    cols  = Dict(h => [v[j] for v in vals] for (j, h) in enumerate(hdr))
    W, _  = read_gal(joinpath(dir, "columbus.gal"); ids = Int.(cols["POLYID"]))
    return cols, W
end

cols, W = read_columbus(joinpath(@__DIR__, "data"))
N = length(cols["CRIME"])

@printf("%d tracts, %d neighbour links, row-standardised: %s\n",
        N, nnz(W), all(isapprox.(sum(W, dims = 2), 1.0)))
@printf("CRIME  mean %6.3f   min %6.3f   max %6.3f\n",
        mean(cols["CRIME"]), minimum(cols["CRIME"]), maximum(cols["CRIME"]))
@printf("INC    mean %6.3f\nHOVAL  mean %6.3f\n", mean(cols["INC"]), mean(cols["HOVAL"]))

## 2. The OLS baseline

Crime on income and housing value. This is the regression every treatment of
these data starts from, and the coefficients below are the published ones.

In [ ]:
y  = cols["CRIME"]
X  = hcat(cols["INC"], cols["HOVAL"])   # fit_spatial adds its own intercept
Xc = hcat(ones(N), X)                   # OLS and lm_tests need it explicitly
vn = ["income", "hoval"]

b_ols = Xc \ y
e_ols = y .- Xc * b_ols

@printf("CRIME = %.3f %+.3f*INC %+.3f*HOVAL\n", b_ols[1], b_ols[2], b_ols[3])
@printf("R2 = %.4f\n", 1 - var(e_ols) / var(y))

Both slopes are negative and the fit is respectable. If we stopped here we would
report that richer neighbourhoods with more valuable housing have less crime, and
we would have no idea whether the standard errors mean anything.

## 3. Is there spatial dependence?

Moran's I on the OLS residuals. Under the null of no spatial autocorrelation its
expectation is $-1/(N-1)$, slightly below zero rather than zero.

In [ ]:
println(moran_global(e_ols, W))

The residuals are clearly autocorrelated. That is a specification warning, not a
diagnosis: it is equally consistent with an omitted spatial lag of `CRIME`, with
spatially correlated errors, and with an omitted variable that happens to be
spatially patterned. Moran's I cannot tell these apart.

## 4. Which model? The LM battery

Anselin's Lagrange Multiplier tests are computed from the OLS residuals and ask
two different questions:

| Test | Null | Alternative |
|---|---|---|
| LM-Lag | $\rho = 0$ | $y = \rho W y + X\beta + \varepsilon$ |
| LM-Error | $\lambda = 0$ | $y = X\beta + u$, $u = \lambda W u + \varepsilon$ |

Each is inflated by the *other* form of dependence, so when both reject - which
is common - they cannot settle the question between them. The robust versions
correct each for the presence of the other, and it is those that decide.

The convention (Anselin and Rey): if exactly one robust test rejects, fit that
model.

In [ ]:
lms = lm_tests(y, Xc, W)
for k in (:lag, :error, :robust_lag, :robust_error)
    println(getfield(lms, k))
end

Read it in order, and resist the urge to want a clean answer.

Both plain tests reject, and the lag statistic is the larger of the two. Among
the robust pair the error test is emphatically not significant, while the robust
lag test lands just *short* of the 5% critical value of 3.84.

So the battery points at a lag process without quite clearing the bar for it.
That is a more useful lesson than a verdict would have been. The robust tests buy
their immunity to the other form of dependence with power, and on 49 observations
that is an expensive purchase. What we have is a significant plain lag test, a
lag statistic larger than the error one, and a robust error test nowhere near
rejecting — three pieces of evidence pointing the same way, and one of them
marginal.

**Fit the lag model**, and expect a referee to ask for the error model as a
robustness check. Section 6 provides it.

## 5. Estimation

Three specifications by maximum likelihood. `ExactLU()` evaluates
$\ln|I - \rho W|$ exactly, which costs nothing at $N = 49$; the Chebyshev
approximation is the default and earns its keep when $N$ runs to thousands.

In [ ]:
m_sar = fit_spatial(SAR(), W, X, y; varnames = vn, method = ExactLU())
m_sdm = fit_spatial(SDM(), W, X, y; varnames = vn, method = ExactLU())
m_sem = fit_spatial_error(W, X, y;   varnames = vn, method = ExactLU())

@printf("SAR  rho    = %+.4f   logL = %8.3f   AIC = %7.2f\n",
        m_sar.rho, loglikelihood(m_sar), spatial_aic(m_sar))
@printf("SDM  rho    = %+.4f   logL = %8.3f   AIC = %7.2f\n",
        m_sdm.rho, loglikelihood(m_sdm), spatial_aic(m_sdm))
@printf("SEM  lambda = %+.4f   logL = %8.3f   AIC = %7.2f\n",
        m_sem.lambda, loglikelihood(m_sem), spatial_aic(m_sem))

In [ ]:
println(regression_table(m_sar, m_sem; model_names = ["SAR", "SEM"]))

## 6. Model comparison

AIC and BIC across the three, then a Vuong test between the two non-nested
candidates the battery left standing.

In [ ]:
println(model_comparison_table(m_sar, m_sdm, m_sem; names = ["SAR", "SDM", "SEM"]))
println(vuong_test(m_sar, m_sem))

AIC prefers SAR, and the Durbin specification does not earn its two extra
parameters. The Vuong test calls SAR and SEM indistinguishable, which is worth
noticing: information criteria will always name a winner, including between
models the data barely separates. The robust LM battery is the sharper
instrument here, and it pointed at the lag.

## 7. Why the coefficients are not the effects

In $y = \rho W y + X\beta + \varepsilon$ the reduced form is
$y = (I - \rho W)^{-1}(X\beta + \varepsilon)$. A change in $x_i$ moves $y_i$,
which moves $y_j$ for every neighbour $j$, which feeds back into $y_i$. So
$\beta$ is not $\partial y_i / \partial x_i$.

LeSage and Pace decompose the total derivative into a **direct** effect (the own
observation, including feedback that leaves and returns), an **indirect** or
spillover effect (everyone else), and their **total**.

In [ ]:
println(spatial_effects(m_sar))

Compare the direct income effect with the raw coefficient in the table above: the
reduced form amplifies it. The indirect effect is roughly two thirds of the
direct one and is statistically significant, so a neighbourhood's crime rate
responds to its neighbours' incomes nearly as much as to its own. That spillover
is the substantive finding, and it is invisible in the coefficient table.

## 8. Exercises

1. Refit with `method = ChebyshevApprox(order = 12)` and compare $\hat\rho$ with
   the `ExactLU()` value. How large is the approximation error, and would it
   change any conclusion reached above?
2. `OPEN` (open space) and `PLUMB` (housing units without full plumbing) are in
   the CSV. Add them and re-run the battery. Does the diagnosis survive a richer
   mean specification?
3. Build a $k$-nearest-neighbour matrix from the centroids with
   `knn_weights(hcat(cols["X"], cols["Y"]), 5)` and redo the analysis. How much
   of the result is a property of the data, and how much of the weight matrix?
4. The `NSA` dummy splits the city north and south. Fit
   `fit_spatial_regime(W, X, y, ExogenousRegimes(Int.(cols["NSA"]) .+ 1))` and
   compare it against the pooled SAR with `regime_chow_test`. The two regimes
   produce visibly different slopes; does the likelihood ratio agree that they
   differ? (Note `spatial_chow_test` is a *panel* test for a break at a point in
   time, and is not the tool for this.)
5. Fit the SEM anyway and compare its $\hat\lambda$ with the SAR $\hat\rho$. The
   two are not comparable in magnitude. Explain why.